In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
env_name = "dev"

spark.sql(f"""
CREATE DATABASE IF NOT EXISTS {env_name}_metadata_db
""")

print("Metadata Database Created")

StatementMeta(, 423a3b11-b98c-40cd-8323-cccf89f3fd86, 3, Finished, Available, Finished, False)

Metadata Database Created


In [2]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {env_name}_metadata_db.control_table
(
    TableName STRING,
    PrimaryKey STRING,
    LoadType STRING,
    WatermarkColumn STRING,
    IsActive STRING
)
USING DELTA
""")

print("Control Table Created")

StatementMeta(, 423a3b11-b98c-40cd-8323-cccf89f3fd86, 4, Finished, Available, Finished, False)

Control Table Created


In [3]:
spark.sql(f"""
INSERT INTO {env_name}_metadata_db.control_table VALUES
('customers','CustomerID','Full','InsertTimestamp','Yes'),
('orders','OrderID','Incremental','InsertTimestamp','Yes'),
('products','ProductID','Full','InsertTimestamp','Yes'),
('order_items','OrderItemID','Incremental','InsertTimestamp','Yes')
""")

print("Metadata Loaded")

StatementMeta(, 423a3b11-b98c-40cd-8323-cccf89f3fd86, 5, Finished, Available, Finished, False)

Metadata Loaded


In [4]:
spark.sql(f"""
SELECT *
FROM {env_name}_metadata_db.control_table
""").show(truncate=False)

StatementMeta(, 423a3b11-b98c-40cd-8323-cccf89f3fd86, 6, Finished, Available, Finished, False)

+-----------+-----------+-----------+---------------+--------+
|TableName  |PrimaryKey |LoadType   |WatermarkColumn|IsActive|
+-----------+-----------+-----------+---------------+--------+
|order_items|OrderItemID|Incremental|InsertTimestamp|Yes     |
|orders     |OrderID    |Incremental|InsertTimestamp|Yes     |
|customers  |CustomerID |Full       |InsertTimestamp|Yes     |
|products   |ProductID  |Full       |InsertTimestamp|Yes     |
+-----------+-----------+-----------+---------------+--------+



In [5]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {env_name}_metadata_db.watermark_table
(
    TableName STRING,
    LastLoadTimestamp TIMESTAMP
)
USING DELTA
""")

print("Watermark Table Created Successfully")

StatementMeta(, 423a3b11-b98c-40cd-8323-cccf89f3fd86, 7, Finished, Available, Finished, False)

Watermark Table Created Successfully


In [6]:
spark.sql(f"""
INSERT INTO {env_name}_metadata_db.watermark_table VALUES
('customers', TIMESTAMP('1900-01-01 00:00:00')),
('orders', TIMESTAMP('1900-01-01 00:00:00')),
('products', TIMESTAMP('1900-01-01 00:00:00')),
('order_items', TIMESTAMP('1900-01-01 00:00:00'))
""")

print("Initial Watermarks Inserted")

StatementMeta(, 423a3b11-b98c-40cd-8323-cccf89f3fd86, 8, Finished, Available, Finished, False)

Initial Watermarks Inserted


In [7]:
spark.sql(f"""
SELECT *
FROM {env_name}_metadata_db.watermark_table
""").show(truncate=False)

StatementMeta(, 423a3b11-b98c-40cd-8323-cccf89f3fd86, 9, Finished, Available, Finished, False)

+-----------+-------------------+
|TableName  |LastLoadTimestamp  |
+-----------+-------------------+
|products   |1900-01-01 00:00:00|
|order_items|1900-01-01 00:00:00|
|customers  |1900-01-01 00:00:00|
|orders     |1900-01-01 00:00:00|
+-----------+-------------------+



In [8]:
primary_keys = {
    "customers": "CustomerID",
    "orders": "OrderID",
    "products": "ProductID",
    "order_items": "OrderItemID"
}

StatementMeta(, 423a3b11-b98c-40cd-8323-cccf89f3fd86, 10, Finished, Available, Finished, False)

In [9]:
metadata_df = spark.sql(f"""
SELECT *
FROM {env_name}_metadata_db.control_table
WHERE IsActive = 'Yes'
""")

metadata_df.show(truncate=False)

StatementMeta(, 423a3b11-b98c-40cd-8323-cccf89f3fd86, 11, Finished, Available, Finished, False)

+-----------+-----------+-----------+---------------+--------+
|TableName  |PrimaryKey |LoadType   |WatermarkColumn|IsActive|
+-----------+-----------+-----------+---------------+--------+
|order_items|OrderItemID|Incremental|InsertTimestamp|Yes     |
|orders     |OrderID    |Incremental|InsertTimestamp|Yes     |
|customers  |CustomerID |Full       |InsertTimestamp|Yes     |
|products   |ProductID  |Full       |InsertTimestamp|Yes     |
+-----------+-----------+-----------+---------------+--------+



In [10]:
metadata = {}

for row in metadata_df.collect():

    metadata[row["TableName"]] = {
        "PrimaryKey": row["PrimaryKey"],
        "LoadType": row["LoadType"],
        "WatermarkColumn": row["WatermarkColumn"]
    }

print(metadata)

StatementMeta(, 423a3b11-b98c-40cd-8323-cccf89f3fd86, 12, Finished, Available, Finished, False)

{'order_items': {'PrimaryKey': 'OrderItemID', 'LoadType': 'Incremental', 'WatermarkColumn': 'InsertTimestamp'}, 'orders': {'PrimaryKey': 'OrderID', 'LoadType': 'Incremental', 'WatermarkColumn': 'InsertTimestamp'}, 'customers': {'PrimaryKey': 'CustomerID', 'LoadType': 'Full', 'WatermarkColumn': 'InsertTimestamp'}, 'products': {'PrimaryKey': 'ProductID', 'LoadType': 'Full', 'WatermarkColumn': 'InsertTimestamp'}}


In [11]:
watermark_df = spark.sql(f"""
SELECT *
FROM {env_name}_metadata_db.watermark_table
""")

watermark_df.show()

StatementMeta(, 423a3b11-b98c-40cd-8323-cccf89f3fd86, 14, Finished, Available, Finished, False)

+-----------+-------------------+
|  TableName|  LastLoadTimestamp|
+-----------+-------------------+
|   products|1900-01-01 00:00:00|
|order_items|1900-01-01 00:00:00|
|  customers|1900-01-01 00:00:00|
|     orders|1900-01-01 00:00:00|
+-----------+-------------------+



In [12]:
watermarks = {}

for row in watermark_df.collect():
    watermarks[row["TableName"]] = row["LastLoadTimestamp"]

print(watermarks)

StatementMeta(, 423a3b11-b98c-40cd-8323-cccf89f3fd86, 16, Finished, Available, Finished, False)

{'products': datetime.datetime(1900, 1, 1, 0, 0), 'order_items': datetime.datetime(1900, 1, 1, 0, 0), 'customers': datetime.datetime(1900, 1, 1, 0, 0), 'orders': datetime.datetime(1900, 1, 1, 0, 0)}
